In [13]:
import os
import pandas as pd
import numpy as np
def read_all_m3_csv_files(data_dir):
    """Read all M3 CSV files and combine into long format"""
    all_data = []
    
    for i in range(1, 1429):  # T1 to T1428
        filename = f"T{i}.csv"
        filepath = os.path.join(data_dir, filename)
        
        if os.path.exists(filepath):
            try:
                df = pd.read_csv(filepath)
                
                # Add series ID
                df['id'] = f"T{i:06d}"  # T000001, T000002, etc.
                
                # Rename columns to match your format
                df = df.rename(columns={
                    'date': 'timestamp',
                    'OT': 'target'
                })
                
                # Convert timestamp to datetime
                df['timestamp'] = pd.to_datetime(df['timestamp'])
                
                # Keep only the columns we need
                df = df[['id', 'timestamp', 'target']]
                
                all_data.append(df)
                
                if i % 100 == 0:  # Progress indicator
                    print(f"Processed {i} files...")
                    
            except Exception as e:
                print(f"Error reading {filename}: {e}")
    
    # Combine all series into one DataFrame
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        return combined_df
    else:
        print("No data was successfully read")
        return pd.DataFrame()

# Read all CSV files
data_dir = "/home/oliver1024/Documents/paper_writing/code_base/inference/m3_yearly_validation"
df_long = read_all_m3_csv_files(data_dir)

print(f"Combined data shape: {df_long.shape}")
print(f"Number of unique series: {df_long['id'].nunique()}")
print(f"Date range: {df_long['timestamp'].min()} to {df_long['timestamp'].max()}")
print("\nFirst few rows:")
print(df_long.head())

Processed 100 files...
Processed 200 files...
Processed 300 files...
Processed 400 files...
Processed 500 files...
Processed 600 files...
Combined data shape: (18319, 3)
Number of unique series: 645
Date range: 1811-01-01 00:00:00 to 2001-01-01 00:00:00

First few rows:
        id  timestamp   target
0  T000001 1975-01-01   940.66
1  T000001 1976-01-01  1084.86
2  T000001 1977-01-01  1244.98
3  T000001 1978-01-01  1445.02
4  T000001 1979-01-01  1683.17


In [14]:
import os
import time
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsforecast import StatsForecast
from statsforecast.models import AutoETS, AutoARIMA, SeasonalNaive, Naive, AutoTheta

from eval_metrics import naive_MASE, seasonal_MASE, SMAPE


# ---------- helpers ----------
def _ensure_dirs(base_dir: str) -> str:
    os.makedirs(base_dir, exist_ok=True)
    plots_dir = os.path.join(base_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)
    return plots_dir

def _supports_kw(model_cls, kw: str) -> bool:
    try:
        return kw in inspect.signature(model_cls.__init__).parameters
    except Exception:
        return False

def _series_label(i: int) -> str:
    return f"T{i}"

def _write_report(path, label, model_name, season_length, y_true, y_pred, ds_true,
                  naive_mase, seasonal_mase, smape, train_t, predict_t, total_t):
    lines = [
        "Scaled Metrics:",
        f"  Naive MASE:     {naive_mase:.4f}",
        f"  Seasonal MASE:  {seasonal_mase:.4f} (seasonality={season_length})",
        f"  SMAPE:          {smape:.4f}%",
        "",
        "Timing (seconds):",
        f"  Train:    {train_t:.2f}s",
        f"  Predict:  {predict_t:.2f}s",
        f"  Total:    {total_t:.2f}s",
        "",
        "="*70,
        f"DETAILED PREDICTIONS  |  Model: {model_name}",
        "="*70,
        "Date                     Actual  Predicted      Error    Error %",
        "-"*70
    ]
    for dt, yt, yp in zip(ds_true, y_true, y_pred):
        err = yp - yt
        err_pct = abs(err) / (abs(yt) + 1e-12) * 100
        lines.append(f"{pd.to_datetime(dt).date():<24} "
                     f"{yt:10.2f}  {yp:10.2f}  {err:10.2f}    {err_pct:6.1f}%")
    lines.append("="*70)
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

def _save_plot(plots_dir, label, train_ts, train_y, test_ts, test_y, pred_y,
               smape, seasonal_mase, split_ts):
    plt.figure(figsize=(12, 4))
    plt.plot(train_ts, train_y, '-', linewidth=2, color=(0.5, 0.5, 0.5), label='Train')
    plt.plot(test_ts,  test_y,  '-', linewidth=3, color='black', label='True (test)')
    plt.plot(test_ts,  pred_y,  '-', linewidth=3, color='magenta', label='Predicted')
    plt.axvline(pd.to_datetime(train_ts.iloc[-1]), color='brown', linestyle='--', linewidth=2)
    plt.title(f"{label} | sMAPE={smape:.2f}%  |  Seasonal MASE={seasonal_mase:.3f}")
    plt.xlabel("Date"); plt.ylabel("Value"); plt.legend(); plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, f"{label}.png"), dpi=150, bbox_inches="tight")
    plt.close()


# ---------- main ----------
def train_forecast_and_evaluate(
    df_long: pd.DataFrame,
    model_class,
    prediction_length: int = 18,
    season_length: int = 12,
    freq: str = "ME",
    base_output_dir: str = "./autoETS/m3_monthly",
    save_txt: bool = True,
    save_plots: bool = True
):
    """
    Runs ONE chosen StatsForecast model.
    Outputs go directly into `base_output_dir`:
      - T1.txt … Tn.txt, average_metrics.txt
      - plots/Tk.png
    """
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'

    model_name = model_class.__name__
    plots_dir = os.path.join(base_output_dir, "plots")
    if save_txt or save_plots:
        os.makedirs(base_output_dir, exist_ok=True)
    if save_plots:
        os.makedirs(plots_dir, exist_ok=True)

    series_ids = df_long['id'].unique()
    results = []

    # Warm-up, discarded. statsforecast is numba-JIT'd, so the first series pays
    # compilation on top of its own fit -- start-up cost, not per-series
    # inference cost, but it lands entirely in series 1's train_t/predict_t and
    # so in the means reported below. pytorch_version/inference.py discards one
    # predict call for the same reason (lazy cuda init there); discarding one
    # full fit+predict here is what makes the two sides' timings comparable.
    for _warm_id in series_ids[:1]:
        _w = df_long[df_long['id'] == _warm_id].sort_values('timestamp')
        if len(_w) > prediction_length:
            _wtrain = _w.iloc[:-prediction_length].rename(
                columns={'id': 'unique_id', 'timestamp': 'ds', 'target': 'y'})
            _wmodel = (model_class(season_length=season_length)
                       if _supports_kw(model_class, "season_length")
                       else model_class())
            _wsf = StatsForecast(models=[_wmodel], freq=freq, n_jobs=1)
            _wsf.fit(_wtrain)
            _wsf.predict(h=prediction_length)

    for idx, series_id in enumerate(series_ids, start=1):
        label = _series_label(idx)
        try:
            sdata = df_long[df_long['id'] == series_id].sort_values('timestamp').copy()
            if len(sdata) <= prediction_length:
                continue

            train = sdata.iloc[:-prediction_length].copy()
            test  = sdata.iloc[-prediction_length:].copy()

            train_sf = train.rename(columns={'id': 'unique_id', 'timestamp': 'ds', 'target': 'y'})

            # model init
            if _supports_kw(model_class, "season_length"):
                model = model_class(season_length=season_length)
            else:
                model = model_class()

            sf = StatsForecast(models=[model], freq=freq, n_jobs=1)

            # perf_counter, not time(): monotonic, and the same clock
            # pytorch_version/inference.py uses, so the two sides' per-series
            # timings are produced by the same instrument.
            total_start = time.perf_counter()

            # train
            t0 = time.perf_counter()
            sf.fit(train_sf)
            train_t = time.perf_counter() - t0

            # predict
            t1 = time.perf_counter()
            fcst = sf.predict(h=prediction_length)
            predict_t = time.perf_counter() - t1

            total_t = time.perf_counter() - total_start

            fc_cols = [c for c in fcst.columns if c not in ("unique_id", "ds")]
            if not fc_cols:
                raise RuntimeError("No forecast column found.")
            y_pred = fcst[fc_cols[0]].to_numpy()
            y_true = test['target'].to_numpy()
            train_true = train['target'].to_numpy()  # <— ADD THIS
            nmase = naive_MASE(y_pred, y_true, train_true)  # <— CHANGED
            smase = seasonal_MASE(y_pred, y_true, train_true, season_length)  # <— CHANGED
            smape = SMAPE(y_pred, y_true)

            # save txt + plot (toggleable)
            if save_txt:
                _write_report(
                    os.path.join(base_output_dir, f"{label}.txt"),
                    label, model_name, season_length,
                    y_true, y_pred, test['timestamp'].to_numpy(),
                    nmase, smase, smape,
                    train_t, predict_t, total_t
                )
            if save_plots:
                _save_plot(
                    plots_dir, label,
                    train['timestamp'], train['target'],
                    test['timestamp'],  test['target'], y_pred,
                    smape, smase, test['timestamp'].iloc[0]
                )

            results.append({
                "series_id": label,
                "naive_mase": nmase,
                "seasonal_mase": smase,
                "smape": smape,
                "train_time": train_t,
                "predict_time": predict_t,
                "total_time": total_t
            })

            print(f"{idx:>4} | {label:<6} {model_name:<13} "
                  f"SMAPE={smape:.3f}%  Seasonal MASE={smase:.3f} Naive MASE={nmase:.3f}  Time={total_t:.4f}s")

        except Exception as e:
            print(f"Error on {label}: {e}")
            continue

    # average metrics
    df = pd.DataFrame(results)
    if not df.empty:
        avg_path = os.path.join(base_output_dir, "average_metrics.txt")
        content = "\n".join([
            "="*70,
            "AVERAGE SCALED METRICS",
            "="*70,
            f"  Naive MASE:     {df['naive_mase'].mean():.4f}",
            f"  Seasonal MASE:  {df['seasonal_mase'].mean():.4f} (seasonality={season_length})",
            f"  SMAPE:          {df['smape'].mean():.4f}%",
            "",
            "Average Timing (seconds):",
            f"  Train:    {df['train_time'].mean():.4f}s",
            f"  Predict:  {df['predict_time'].mean():.4f}s",
            f"  Total:    {df['total_time'].mean():.4f}s",
            "="*70
        ])

        # write file (toggleable)
        if save_txt:
            with open(avg_path, "w", encoding="utf-8") as f:
                f.write(content)

        # always print content to output
        print("\n" + "="*70)
        print("AVERAGE METRICS FILE CONTENT:\n")
        print(content)
        print("="*70 + "\n")

    return df





In [15]:
# =========================
# Configurable section
# =========================
PREDICTION_LENGTH = 6
SEASONALITY       = 1
FREQ              = "YE"

# 👇 You manually set this each time you switch models
BASE_DIR          = "./autoTheta/m3_yearly"   # e.g. "./autoARIMA/m3_monthly"

# 👇 Choose ONE model
MODEL = AutoTheta   # or AutoARIMA / SeasonalNaive / Naive

results_df = train_forecast_and_evaluate(
    df_long,
    model_class=MODEL,
    prediction_length=PREDICTION_LENGTH,
    season_length=SEASONALITY,
    freq=FREQ,
    base_output_dir=BASE_DIR,
    save_txt=True,     # 👈 toggle per-series .txt reports
    save_plots=True    # 👈 toggle per-series .png plots
)

   1 | T1     AutoTheta     SMAPE=18.783%  Seasonal MASE=4.339 Naive MASE=4.339  Time=0.0033s
   2 | T2     AutoTheta     SMAPE=7.796%  Seasonal MASE=0.723 Naive MASE=0.723  Time=0.0024s
   3 | T3     AutoTheta     SMAPE=16.827%  Seasonal MASE=1.076 Naive MASE=1.076  Time=0.0025s
   4 | T4     AutoTheta     SMAPE=8.733%  Seasonal MASE=0.769 Naive MASE=0.769  Time=0.0027s
   5 | T5     AutoTheta     SMAPE=9.822%  Seasonal MASE=0.722 Naive MASE=0.722  Time=0.0035s
   6 | T6     AutoTheta     SMAPE=9.831%  Seasonal MASE=1.689 Naive MASE=1.689  Time=0.0029s
   7 | T7     AutoTheta     SMAPE=13.361%  Seasonal MASE=1.572 Naive MASE=1.572  Time=0.0030s
   8 | T8     AutoTheta     SMAPE=23.693%  Seasonal MASE=3.524 Naive MASE=3.524  Time=0.0030s
   9 | T9     AutoTheta     SMAPE=18.460%  Seasonal MASE=1.390 Naive MASE=1.390  Time=0.0028s
  10 | T10    AutoTheta     SMAPE=64.461%  Seasonal MASE=9.190 Naive MASE=9.190  Time=0.0028s
  11 | T11    AutoTheta     SMAPE=4.370%  Seasonal MASE=0.860 Na

# AutoETS

In [16]:
# =========================
# Configurable section
# =========================
PREDICTION_LENGTH = 6
SEASONALITY       = 1
FREQ              = "YE"

# 👇 You manually set this each time you switch models
BASE_DIR          = "./autoETS/m3_yearly"   # e.g. "./autoARIMA/m3_monthly"

# 👇 Choose ONE model
MODEL = AutoETS   # or AutoARIMA / SeasonalNaive / Naive

results_df = train_forecast_and_evaluate(
    df_long,
    model_class=MODEL,
    prediction_length=PREDICTION_LENGTH,
    season_length=SEASONALITY,
    freq=FREQ,
    base_output_dir=BASE_DIR,
    save_txt=True,     # 👈 toggle per-series .txt reports
    save_plots=True    # 👈 toggle per-series .png plots
)

   1 | T1     AutoETS       SMAPE=22.474%  Seasonal MASE=5.079 Naive MASE=5.079  Time=0.0053s
   2 | T2     AutoETS       SMAPE=19.153%  Seasonal MASE=1.698 Naive MASE=1.698  Time=0.0038s
   3 | T3     AutoETS       SMAPE=6.402%  Seasonal MASE=0.375 Naive MASE=0.375  Time=0.0035s
   4 | T4     AutoETS       SMAPE=13.436%  Seasonal MASE=1.240 Naive MASE=1.240  Time=0.0035s
   5 | T5     AutoETS       SMAPE=17.930%  Seasonal MASE=1.401 Naive MASE=1.401  Time=0.0038s
   6 | T6     AutoETS       SMAPE=5.075%  Seasonal MASE=0.836 Naive MASE=0.836  Time=0.0038s
   7 | T7     AutoETS       SMAPE=2.858%  Seasonal MASE=0.312 Naive MASE=0.312  Time=0.0037s
   8 | T8     AutoETS       SMAPE=9.177%  Seasonal MASE=1.205 Naive MASE=1.205  Time=0.0037s
   9 | T9     AutoETS       SMAPE=33.353%  Seasonal MASE=2.352 Naive MASE=2.352  Time=0.0034s
  10 | T10    AutoETS       SMAPE=63.625%  Seasonal MASE=8.958 Naive MASE=8.958  Time=0.0037s
  11 | T11    AutoETS       SMAPE=4.733%  Seasonal MASE=0.945 Na

# Auto ARIMA

In [17]:
# =========================
# Configurable section
# =========================
PREDICTION_LENGTH = 6
SEASONALITY       = 1
FREQ              = "YE"

# 👇 You manually set this each time you switch models
BASE_DIR          = "./autoARIMA/m3_yearly"   # e.g. "./autoARIMA/m3_monthly"

# 👇 Choose ONE model
MODEL = AutoARIMA   # or AutoETS/AutoARIMA / SeasonalNaive / Naive

results_df = train_forecast_and_evaluate(
    df_long,
    model_class=MODEL,
    prediction_length=PREDICTION_LENGTH,
    season_length=SEASONALITY,
    freq=FREQ,
    base_output_dir=BASE_DIR,
    save_txt=True,     # 👈 toggle per-series .txt reports
    save_plots=True    # 👈 toggle per-series .png plots
)

   1 | T1     AutoARIMA     SMAPE=6.260%  Seasonal MASE=1.567 Naive MASE=1.567  Time=0.0452s
   2 | T2     AutoARIMA     SMAPE=19.152%  Seasonal MASE=1.698 Naive MASE=1.698  Time=0.0724s
   3 | T3     AutoARIMA     SMAPE=6.403%  Seasonal MASE=0.375 Naive MASE=0.375  Time=0.0820s
   4 | T4     AutoARIMA     SMAPE=11.149%  Seasonal MASE=1.012 Naive MASE=1.012  Time=0.1454s
   5 | T5     AutoARIMA     SMAPE=18.333%  Seasonal MASE=1.436 Naive MASE=1.436  Time=0.1897s
   6 | T6     AutoARIMA     SMAPE=17.223%  Seasonal MASE=3.142 Naive MASE=3.142  Time=0.0987s
   7 | T7     AutoARIMA     SMAPE=2.858%  Seasonal MASE=0.312 Naive MASE=0.312  Time=0.1023s
   8 | T8     AutoARIMA     SMAPE=26.480%  Seasonal MASE=4.026 Naive MASE=4.026  Time=0.1203s
   9 | T9     AutoARIMA     SMAPE=33.351%  Seasonal MASE=2.352 Naive MASE=2.352  Time=0.0846s
  10 | T10    AutoARIMA     SMAPE=63.631%  Seasonal MASE=8.959 Naive MASE=8.959  Time=0.0804s
  11 | T11    AutoARIMA     SMAPE=5.730%  Seasonal MASE=1.181 N

# Seasonal Naive

In [18]:
# =========================
# Configurable section
# =========================
PREDICTION_LENGTH = 6
SEASONALITY       = 1
FREQ              = "YE"

# 👇 You manually set this each time you switch models
BASE_DIR          = "./seasonal_naive/m3_yearly"   # e.g. "./autoARIMA/m3_monthly"

# 👇 Choose ONE model
MODEL = SeasonalNaive   # or AutoETS/AutoARIMA / SeasonalNaive / Naive

results_df = train_forecast_and_evaluate(
    df_long,
    model_class=MODEL,
    prediction_length=PREDICTION_LENGTH,
    season_length=SEASONALITY,
    freq=FREQ,
    base_output_dir=BASE_DIR,
    save_txt=True,     # 👈 toggle per-series .txt reports
    save_plots=True    # 👈 toggle per-series .png plots
)

   1 | T1     SeasonalNaive SMAPE=36.820%  Seasonal MASE=7.704 Naive MASE=7.704  Time=0.0022s
   2 | T2     SeasonalNaive SMAPE=19.152%  Seasonal MASE=1.698 Naive MASE=1.698  Time=0.0026s
   3 | T3     SeasonalNaive SMAPE=6.403%  Seasonal MASE=0.375 Naive MASE=0.375  Time=0.0027s
   4 | T4     SeasonalNaive SMAPE=10.811%  Seasonal MASE=0.868 Naive MASE=0.868  Time=0.0020s
   5 | T5     SeasonalNaive SMAPE=17.932%  Seasonal MASE=1.401 Naive MASE=1.401  Time=0.0020s
   6 | T6     SeasonalNaive SMAPE=4.294%  Seasonal MASE=0.707 Naive MASE=0.707  Time=0.0021s
   7 | T7     SeasonalNaive SMAPE=2.858%  Seasonal MASE=0.312 Naive MASE=0.312  Time=0.0020s
   8 | T8     SeasonalNaive SMAPE=9.178%  Seasonal MASE=1.205 Naive MASE=1.205  Time=0.0020s
   9 | T9     SeasonalNaive SMAPE=33.351%  Seasonal MASE=2.352 Naive MASE=2.352  Time=0.0022s
  10 | T10    SeasonalNaive SMAPE=52.723%  Seasonal MASE=6.112 Naive MASE=6.112  Time=0.0021s
  11 | T11    SeasonalNaive SMAPE=15.440%  Seasonal MASE=2.881 N

# Naive

In [19]:
# =========================
# Configurable section
# =========================
PREDICTION_LENGTH = 6
SEASONALITY       = 1
FREQ              = "YE"

# 👇 You manually set this each time you switch models
BASE_DIR          = "./naive/m3_yearly"   # e.g. "./autoARIMA/m3_monthly"

# 👇 Choose ONE model
MODEL = Naive   # or AutoETS/AutoARIMA / SeasonalNaive / Naive

results_df = train_forecast_and_evaluate(
    df_long,
    model_class=MODEL,
    prediction_length=PREDICTION_LENGTH,
    season_length=SEASONALITY,
    freq=FREQ,
    base_output_dir=BASE_DIR,
    save_txt=True,     # 👈 toggle per-series .txt reports
    save_plots=True    # 👈 toggle per-series .png plots
)

   1 | T1     Naive         SMAPE=36.820%  Seasonal MASE=7.704 Naive MASE=7.704  Time=0.0023s
   2 | T2     Naive         SMAPE=19.152%  Seasonal MASE=1.698 Naive MASE=1.698  Time=0.0024s
   3 | T3     Naive         SMAPE=6.403%  Seasonal MASE=0.375 Naive MASE=0.375  Time=0.0020s
   4 | T4     Naive         SMAPE=10.811%  Seasonal MASE=0.868 Naive MASE=0.868  Time=0.0021s
   5 | T5     Naive         SMAPE=17.932%  Seasonal MASE=1.401 Naive MASE=1.401  Time=0.0020s
   6 | T6     Naive         SMAPE=4.294%  Seasonal MASE=0.707 Naive MASE=0.707  Time=0.0025s
   7 | T7     Naive         SMAPE=2.858%  Seasonal MASE=0.312 Naive MASE=0.312  Time=0.0021s
   8 | T8     Naive         SMAPE=9.178%  Seasonal MASE=1.205 Naive MASE=1.205  Time=0.0022s
   9 | T9     Naive         SMAPE=33.351%  Seasonal MASE=2.352 Naive MASE=2.352  Time=0.0022s
  10 | T10    Naive         SMAPE=52.723%  Seasonal MASE=6.112 Naive MASE=6.112  Time=0.0022s
  11 | T11    Naive         SMAPE=15.440%  Seasonal MASE=2.881 N